In [16]:
import torch
from torch import nn
import numpy as np

X = torch.tensor([[0., 0],
                  [0., 1.],
                  [1., 0.],
                  [1., 1.]])

y_and = torch.tensor([[0.], [0.], [0.], [1.]])
y_or  = torch.tensor([[0.], [1.], [1.], [1.]])


model = nn.Sequential(
    nn.Linear(2, 1),
    nn.ReLU()
)

criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

In [17]:
for epoch in range(1000):
    optimizer.zero_grad()
    
    y_pred = model(X)
    loss = criterion(y_pred, y_and)
    
    loss.backward()
    optimizer.step()

    if epoch % 200 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 0.3161
Epoch 200, Loss: 0.0046
Epoch 400, Loss: 0.0001
Epoch 600, Loss: 0.0000
Epoch 800, Loss: 0.0000


In [18]:
for x in model.parameters():
    print(x)

Parameter containing:
tensor([[0.9998, 0.9998]], requires_grad=True)
Parameter containing:
tensor([-0.9998], requires_grad=True)


We got W = [1, 1] and b = -1, which is practically a canonical AND representation.

## My implementation

In [20]:
import numpy as np

from module.linear import LinearLayer
from module.relu import ReluLayer
from module.network import NeuralNetwork
from optimizer.sgd import SGD
from criterion.mse import MSELoss


X = np.array([[0., 0],
                  [0., 1.],
                  [1., 0.],
                  [1., 1.]])

y_and = np.array([[0.], [0.], [0.], [1.]])
y_or  = np.array([[0.], [1.], [1.], [1.]])
y_first  = np.array([[0.], [0.], [1.], [1.]])
y_second  = np.array([[1.], [0.], [1.], [0.]])

model = NeuralNetwork([
    LinearLayer(2,1, init="he"),
    ReluLayer()
])

criterion = MSELoss()
optimizer = SGD(model.parameters, lr=0.1)


for epoch in range(2000):
    y_pred = model.forward(X)

    loss = criterion.forward(y_and, y_pred)
    
    delta = criterion.backward(y_and, y_pred)
    model.backward(delta)
    
    optimizer.step()

    if epoch % 200 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

params = [p for p, g in model.parameters()]
W = params[0]
b = params[1]

print(W, b)


Epoch 0, Loss: 0.2308
Epoch 200, Loss: 0.0034
Epoch 400, Loss: 0.0001
Epoch 600, Loss: 0.0000
Epoch 800, Loss: 0.0000
Epoch 1000, Loss: 0.0000
Epoch 1200, Loss: 0.0000
Epoch 1400, Loss: 0.0000
Epoch 1600, Loss: 0.0000
Epoch 1800, Loss: 0.0000
[[0.99999997]
 [0.99999997]] [-0.99999996]


We got W = [1, 1] and b = -1, which is the same as torch!

## Majority with 3 inputs

In [42]:
X = np.array([[0,0,0],
              [0,0,1],
              [0,1,0],
              [0,1,1],
              [1,0,0],
              [1,0,1],
              [1,1,0],
              [1,1,1]])

y_majority = np.array([[0],[0],[0],[1],[0],[1],[1],[1]])

model = NeuralNetwork([
    LinearLayer(3,1, init="he"),
    ReluLayer()
])

criterion = MSELoss()
optimizer = SGD(model.parameters, lr=0.1)


for epoch in range(2000):
    y_pred = model.forward(X)

    loss = criterion.forward(y_majority, y_pred)
    
    delta = criterion.backward(y_majority, y_pred)
    model.backward(delta)
    
    optimizer.step()

    if epoch % 200 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

params = [p for p, g in model.parameters()]
W = params[0]
b = params[1]

print(W, b)

Epoch 0, Loss: 0.2358
Epoch 200, Loss: 0.0470
Epoch 400, Loss: 0.0469
Epoch 600, Loss: 0.0469
Epoch 800, Loss: 0.0469
Epoch 1000, Loss: 0.0469
Epoch 1200, Loss: 0.0469
Epoch 1400, Loss: 0.0469
Epoch 1600, Loss: 0.0469
Epoch 1800, Loss: 0.0469
[[0.625]
 [0.625]
 [0.625]] [-0.5]


We got W = [0.625, 0.625, 0.625] and b = [-0.5]. That's as good as perceptron with MSE can get!